# Workshop 2B: ตรวจชิ้นงาน OK / NG
โจทย์: ในถาดต้องมีชิ้นสีแดงขนาดใหญ่ **ครบตามจำนวน** ถ้าขาดหรือเกิน = **NG**
ในโรงงาน เซนเซอร์จะสั่งถ่ายภาพเมื่อชิ้นงานมาถึง (Trigger)  ใน Notebook นี้ **รัน cell ข้อ 3 หนึ่งครั้ง = Trigger หนึ่งครั้ง**

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../scripts"))   # ให้หาไฟล์ vision_camera.py เจอ

import cv2
import numpy as np
from vision_camera import show, grab_one, img_path, put_label
print("OpenCV", cv2.__version__, "พร้อมใช้งาน")

## 1. ตั้งค่างานตรวจ

In [ ]:
SOURCE   = "file"     # "file" | "emu" | "basler"
IMAGE    = "shapes.png"
EXPECT   = 5          # จำนวนชิ้นที่ต้องมี
H_LOW, H_HIGH = 0, 10
MIN_AREA = 1000
LOG_FILE = "inspection_log.csv"

## 2. ฟังก์ชันนับและตัดสิน

In [ ]:
def count_big(img):
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    mask = cv2.inRange(hsv, np.array([H_LOW, 50, 50]), np.array([H_HIGH, 255, 255]))
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    boxes = [cv2.boundingRect(c) for c in contours if cv2.contourArea(c) > MIN_AREA]
    return boxes

def judge(n):
    return "OK" if n == EXPECT else "NG"

## 3. Trigger: รัน cell นี้ซ้ำได้เรื่อยๆ
ลองเปลี่ยน `EXPECT` ใน cell ข้อ 1 เป็น 6 แล้วรันใหม่ จะได้ NG

In [ ]:
import csv, time
from datetime import datetime

t0 = time.perf_counter()
frame = grab_one(SOURCE, image=IMAGE)
boxes = count_big(frame)
verdict = judge(len(boxes))
ms = (time.perf_counter() - t0) * 1000

view = frame.copy()
for x, y, w, h in boxes:
    cv2.rectangle(view, (x, y), (x + w, y + h), (200, 0, 100), 3)
color = (0, 160, 0) if verdict == "OK" else (0, 0, 210)
cv2.rectangle(view, (0, 0), (view.shape[1], 50), color, -1)
cv2.putText(view, f"{verdict}  {len(boxes)}/{EXPECT}", (10, 38),
            cv2.FONT_HERSHEY_SIMPLEX, 1.2, (255, 255, 255), 3)

now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
new = not os.path.exists(LOG_FILE)
with open(LOG_FILE, "a", newline="", encoding="utf-8-sig") as f:
    w = csv.writer(f)
    if new:
        w.writerow(["time", "verdict", "count", "expect", "ms"])
    w.writerow([now, verdict, len(boxes), EXPECT, f"{ms:.0f}"])

if verdict == "NG":                                   # เก็บภาพชิ้นงานเสียไว้ตรวจย้อนหลัง
    os.makedirs("ng_images", exist_ok=True)
    cv2.imwrite(f"ng_images/NG_{now.replace(':', '-')}.png", view)

print(f"{now}  ผล: {verdict}  พบ {len(boxes)} ชิ้น (ต้องมี {EXPECT})  ใช้เวลา {ms:.0f} ms")
show(view, titles=[verdict])

## 4. ดูบันทึกผลการตรวจ

In [ ]:
with open(LOG_FILE, encoding="utf-8-sig") as f:
    rows = list(csv.reader(f))[1:]
ok = sum(1 for r in rows if r[1] == "OK")
print(f"ตรวจทั้งหมด {len(rows)} ครั้ง   OK {ok}   NG {len(rows) - ok}")
for r in rows[-5:]:
    print(r)

## ลองเอง (ระดับ 2)
ให้เป็น NG ด้วย ถ้ามี **ชิ้นแดงเล็ก** ปนมา (ชิ้นที่พื้นที่น้อยกว่า `MIN_AREA`)
คำใบ้: นับ contour ทั้งหมด แล้วลบจำนวนชิ้นใหญ่